# What this note book is about

This notebook shows:

* The creation of two **certification authorities**.
* The creation of two subjects.
* The **certification** of those subjects by the certification authorities.
* A **digital signature** created by a subject and verified by the other.


### Warning: password-protected keys

All the commands below access files containing a private key by specifying a password in the command line, for simplicity.

**This is an insecure practice that should never be used in a production environment.** ((unless you really know what you are doing, understand and accept the risk).)

More details in "Protezione chiavi con password" in [this page](https://bartolialberto.github.io/ComputerNetworks/6%20-%20Network%20Security/Approfondimenti_Network_Security/).


### Preparation

In [ ]:
!cd ~/; openssl rand -writerand .rnd

# Subjects: Shevchenko and Van Basten

Create keypairs and self-signed certificates for two named subjects.



In [ ]:
!openssl req -x509 -newkey rsa:2048 -keyout private_key_a.pem -out certificate_a_selfsigned.crt -days 365 -subj "/CN=Andry Shevchenko" -passout pass:password_andry
!openssl req -x509 -newkey rsa:2048 -keyout private_key_m.pem -out certificate_m_selfsigned.crt -days 365 -subj "/CN=Marco Van Basten" -passout pass:password_marco

Dump certificate content in human-readable form (just to make sure they contain what we expect).

In [ ]:
!openssl x509 -in certificate_a_selfsigned.crt  -noout -text
!openssl x509 -in certificate_m_selfsigned.crt  -noout -text

Create the corresponding *Certificate Signing Requests (CSR)*.

These are signed files with standard format that contains a Subject-Public Key pair and that can be given to a certification authority in the certificate request procedure.

The fact that a CSR is signed is useful for proving knowledge of the matching private key.

In [ ]:
!openssl x509 -x509toreq -in certificate_a_selfsigned.crt -signkey private_key_a.pem -out csr_a.pem -passin pass:password_andry
!openssl x509 -x509toreq -in certificate_m_selfsigned.crt -signkey private_key_m.pem -out csr_m.pem -passin pass:password_marco

Dump CSR content in human-readable form (just to make sure they contain what we expect).

In [ ]:
!openssl req -in csr_a.pem -noout -text
!openssl req -in csr_k.pem -noout -text

# Certification Authorities: ACME and Goldrake

Create keypairs and self-signed certificates for two named certification authorities.

*The commands are the same as for any other subject.*

In [ ]:
!openssl req -x509 -newkey rsa:2048 -keyout private_key_c1.pem -out certificate_c1_selfsigned.crt -days 365 -subj "/CN=ACME" -passout pass:password_acme
!openssl req -x509 -newkey rsa:2048 -keyout private_key_c2.pem -out certificate_c2_selfsigned.crt -days 365 -subj "/CN=Goldrake" -passout pass:password_goldrake

Dump certificate content in human-readable form (just to make sure they contain what we expect).

In [ ]:
!openssl x509 -in certificate_c1_selfsigned.crt  -noout -text
!openssl x509 -in certificate_c2_selfsigned.crt  -noout -text

# Certificate Issuance

ACME CA issues a certificate for Shevchenko (`certificate_a.pem`)

In [ ]:
!openssl x509 -req -in csr_a.pem -CA certificate_c1_selfsigned.crt -CAkey private_key_c1.pem -CAcreateserial -out certificate_a.pem -days 365 -passin pass:password_acme
!openssl x509 -in certificate_a.pem  -noout -text


Goldrake CA issues a certificate for Van Basten (`certificate_b.pem`)

In [ ]:
!openssl x509 -req -in csr_m.pem -CA certificate_c2_selfsigned.crt -CAkey private_key_c2.pem -CAcreateserial -out certificate_m.pem -days 365 -passin pass:password_goldrake
!openssl x509 -in certificate_m.pem  -noout -text

# Sign and Verify

## Shevchenko signs...

Shevchenko signs `sample.txt`. The password is needed: why?

The *openssl* command used here outputs signature and certificate of the signer in a separate file (called `signature.p7b`, where p7b is a standard format).



In [ ]:
!echo "This is a sample text file." > sample.txt
!openssl smime -sign -in sample.txt -binary -signer certificate_a.pem -inkey private_key_a.pem -outform PEM -out signature.p7b -passin pass:password_andry

We concatenate signed file (`sample.txt`), signature and certificate (`signature.p7b`) in a single file, that we call `signed_bundle.p7b`.

In [ ]:
!cat sample.txt signature.p7b > signed_bundle.p7b

## ...Van Basten verifies

We assume that Van Basten can use both certification authorities.

We also assume that Keyset and Trustset are contained in the same file for simplicity, that we call `keyset_trustset_vanbasten.pem`.

In [ ]:
!cat certificate_c1_selfsigned.crt certificate_c2_selfsigned.crt > keyset_trustset_vanbasten.pem

Verify `signed_bundle.p7b`, specifying KeySet and TrustSet.

No password is needed: why?

In [ ]:
!echo "Signer is:"
!openssl pkcs7 -inform PEM -in signed_bundle.p7b -print_certs -text | openssl x509 -noout -subject
!echo ""
!openssl smime -verify -inform PEM -in signed_bundle.p7b -CAfile keyset_trustset_vanbasten.pem -content sample.txt


# Play yourself

Try the following:

- Modify some byte in `signed_bundle.p7b` and make sure verification fails ([Linux command](https://chat.openai.com/share/328dfbdb-c4d6-4b04-9fc3-829c98b7cbdb) for modifying one byte, explained by ChatGPT).
- Create a KeySet and TrustSet that contains only the Goldrake CA; then verify the `signed_bundle.p7b` with that KeySet and TrustSet (recall that Shevchenko has been certified by the ACME CA: what verification result do you expect?)
- Swap the roles between the (simulated) subjects: sign as Van Basten, then verify as Shevchenko.

Other playing ideas:

1. Look at one of the self-signed certificates in the KeySet/TrustSet of your PC or of your smartphone. Take note of the tuple that specifies the Subject (CN, ON, C). Create a public-private keypair and self-signed certificate with *exactly* that Subject.
2. Create a public-private keypair for a website of your interest, by specifying its DNS name of CN of the Subject. Then construct a CSR and have that CSR signed by the fake certification authority constructed at step 1.
3. Print the (fake) certificate constructed at step 1; then extract the real self-signed certificate from your PC/smartphone and print that certificate. How can you tell which is the real one?
4. How can you make sure that the self-signed certificates in the KeySet/TrustSet of your PC/smartphone have not been replaced by fake versions constructed as above?
